# Binary Search Trees: Zero to Hero

**NB-07 in the [DSA: Zero to Hero](README.md) series.**

NB-06 kept saying a plain binary tree promises nothing. This notebook adds the one promise that
changes everything — and then measures what happens when nothing enforces it.

***

## Why this notebook is different

- **The invariant is stated precisely, and the usual version is shown to be wrong.** "Left is
  smaller" is not the BST property, and §2.1 builds the four-node tree that satisfies it while
  being a broken BST. The differential test then **finds its own counterexample** in seconds.
- **Degeneration is measured, not warned about.** §3 inserts the same 50,000 keys in three orders
  and counts comparisons per search: **about 15 balanced, 19 random, and 25,000 sorted** — a
  factor of roughly 1,700. Building the tree is $\Theta(n \log n)$ from random input and a
  measured $\Theta(n^2)$ from sorted input. That single table is the entire reason NB-08 exists.
- **Random insertion is measured against its theory.** Average node depth tracks $2\ln n - 3$
  closely, and the expected height's famous $4.311 \ln n$ is approached so slowly that at
  $n = 100{,}000$ the measured constant is still only about 3.4. Quoting the asymptotic without
  checking it would have overstated the height by roughly a quarter.

And the Java section produces the most surprising output in the notebook: a `TreeSet` with a
comparator that orders by string length **silently swallows an element**, then reports
`contains("bbb") == true` for a string it does not contain. Ordered containers decide equality with
`compare() == 0`, never with `equals()`, and §1.4 shows what that costs.

***

## Contents

**Part 1 — Theory from zero**
1. The invariant, stated precisely
2. Search, insert, and the three cases of delete
3. Inorder yields sorted order — the invariant restated
4. Java: `Comparable`, `Comparator`, and the contract that breaks `TreeSet`

**Part 2 — Worked problems** — validating a BST, successor and predecessor, LCA, range queries
**Part 3 — The signature difficulty: nothing enforces balance**
**Part 4 — Tough questions** · **Part 5 — Practice** · **Part 6 — Reading**

***

## In one paragraph

A **binary search tree** is a binary tree carrying one extra promise: for every node, *every* key
in its left subtree is smaller and *every* key in its right subtree is larger. That promise — not
the weaker "its children are ordered", which §2.1 shows is a different and useless property — is
what lets search discard half the remaining tree at each step, giving $O(h)$ search, insert and
delete where a plain tree gives $\Theta(n)$. It also makes **inorder traversal produce sorted
output**, which is the invariant restated rather than a separate fact, and it is what a hash table
(NB-03) cannot do at any price. Insert and search are straightforward; **delete has three cases**
and the third — a node with two children — is the one worth understanding, because it is resolved
by replacing the key with its inorder successor, the smallest key in the right subtree. The catch
is in that $O(h)$: **nothing in the definition bounds $h$.** Insert keys in sorted order and every
node goes to the right of the last, producing a linked list with $h = n - 1$ — and since sorted or
near-sorted input is extremely common, this is not a theoretical worry but the normal failure. §3
measures it, and NB-08's balanced trees are the answer.

**Prerequisites:** [NB-06 Trees & Traversals](trees_zero_to_hero.ipynb) for the traversals — this
notebook uses inorder constantly and its §3 measured why every operation here is written
iteratively — and [NB-03 Hashing](hashing_zero_to_hero.ipynb) for the comparison that decides when
you want a tree rather than a hash table.

***
# Part 0 - Setup

Standard library only, plus `dsa_toolkit`. §1.4 needs the JDK.

**Every operation in this notebook is iterative.** NB-06 §3 measured recursion dying at a
1,000-node degenerate tree, and §3 here builds degenerate trees of 50,000 nodes on purpose — so a
recursive `insert` or `delete` would crash on exactly the input the notebook is about.

In [1]:
# ---------------------------------------------------------------------------
# Everything this notebook uses. Standard library only.
# ---------------------------------------------------------------------------
import math
import random
import statistics
import sys
import time

from dsa_toolkit import (InvariantError, JavaError, StressFailure, check_invariant,
                         cross_check, growth_table, java_available, measure_growth,
                         run_java, stress)

RANDOM_SEED = 12345

ok, detail = java_available()
JAVA = ok
print("python", sys.version.split()[0])
print("JDK available:", ok, "|", detail)

python 3.14.7
JDK available: True | javac 25.0.4.1


***
# Part 1 - Theory from zero

1. The invariant, stated precisely
2. Search, insert, and the three cases of delete
3. Inorder yields sorted order — the invariant restated
4. Java: `Comparable`, `Comparator`, and the contract that breaks `TreeSet`

## 1.1 The invariant, stated precisely

Here is the statement people usually give:

> ~~For every node, its left child is smaller and its right child is larger.~~

**That is not the BST property.** It is a weaker condition that a broken tree can satisfy, and
§2.1 builds one. The correct statement quantifies over whole subtrees:

> **The BST invariant.** For every node $x$: **every** key in $x$'s left subtree is less than
> $x$.key, and **every** key in $x$'s right subtree is greater than $x$.key.

The difference is the difference between *local* and *global*. The weak version constrains a node
against two neighbours; the real one constrains it against everything below it. A grandchild three
levels down still has to respect an ancestor it never touches — and that is precisely the property
search relies on, because search discards an entire subtree on one comparison.

**An equivalent formulation**, which is often easier to work with and is the basis of §2.1's
correct validator:

> Each node is confined to an **open interval** $(lo, hi)$ inherited from its ancestors. The root
> is confined to $(-\infty, +\infty)$; descending left tightens $hi$ to the parent's key, and
> descending right tightens $lo$.

**A third equivalent formulation**, which §1.3 measures: **the inorder traversal is strictly
increasing.** All three say the same thing, and having three ways to say it is genuinely useful —
each suggests a different implementation and a different test.

**On duplicates.** The definition above says *less* and *greater*, so duplicate keys have no home.
Real implementations pick one of: reject them (this notebook), keep a count per node, or send them
consistently one way and weaken one comparison to $\le$. **Pick one and write it down** — mixing
conventions gives a tree where a key exists twice and only one copy is findable.

In [2]:
# ---------------------------------------------------------------------------
# 1.1 The node, and the invariant as a predicate.
# ---------------------------------------------------------------------------
class BNode:
    __slots__ = ("key", "left", "right")

    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None


def in_range(node, lo=None, hi=None):
    """The invariant, as the interval formulation. Iterative, per Part 0."""
    stack = [(node, lo, hi)]
    while stack:
        n, low, high = stack.pop()
        if n is None:
            continue
        if low is not None and n.key <= low:
            return False
        if high is not None and n.key >= high:
            return False
        stack.append((n.left, low, n.key))     # descending left tightens the upper bound
        stack.append((n.right, n.key, high))   # descending right tightens the lower bound
    return True


# The tree everyone's favourite wrong definition accepts.
#        10
#       /  \
#      5    15
#          /
#         6      <- inside 10's RIGHT subtree, but smaller than 10
broken = BNode(10)
broken.left = BNode(5)
broken.right = BNode(15)
broken.right.left = BNode(6)

print("        10")
print("       /  \\")
print("      5    15")
print("          /")
print("         6")
print()
print("  Every node is correctly ordered against its own children:")
print("    5 < 10 < 15, and 6 < 15.")
print("  But 6 sits in 10's right subtree while being less than 10.")
print()
print("  is this a BST, by the interval definition?", in_range(broken))
print()
print("  Search for 6 starting at the root:")
n, path = broken, []
while n is not None:
    path.append(n.key)
    if 6 == n.key:
        break
    n = n.left if 6 < n.key else n.right
print("    visited", path, "-> found:", n is not None)
print("    Search went LEFT at 10 because 6 < 10, and never looked in the")
print("    right subtree where 6 actually is. The tree is unusable.")

        10
       /  \
      5    15
          /
         6

  Every node is correctly ordered against its own children:
    5 < 10 < 15, and 6 < 15.
  But 6 sits in 10's right subtree while being less than 10.

  is this a BST, by the interval definition? False

  Search for 6 starting at the root:
    visited [10, 5] -> found: False
    Search went LEFT at 10 because 6 < 10, and never looked in the
    right subtree where 6 actually is. The tree is unusable.


## 1.2 Search, insert, and the three cases of delete

**Search** is the invariant used directly: compare, and discard the half that cannot contain the
key. $O(h)$ comparisons.

**Insert** is search that does not stop early: walk down as if looking for the key, and when you
fall off the tree, that empty slot is exactly where the key belongs. $O(h)$, and it never
restructures anything — which is precisely the freedom that lets the tree degenerate (§3).

**Delete has three cases**, and this is the operation worth understanding properly:

1. **No children.** Detach it. Nothing else changes.
2. **One child.** Splice the child into the node's place — the child's whole subtree already
   satisfies whatever bound the node satisfied, so the invariant survives untouched.
3. **Two children.** You cannot simply remove it: it has two subtrees and one hole. **Replace its
   key with its inorder successor** — the smallest key in the right subtree — and then delete that
   successor from the right subtree instead.

Case 3 works because the inorder successor is the *only* key that can take the node's place: it is
larger than everything in the left subtree (being in the right one) and smaller than everything
else in the right subtree (being its minimum). And the recursion terminates because the successor,
being a leftmost node, **has no left child** — so deleting it is always case 1 or case 2, never
case 3 again.

The predecessor (largest in the left subtree) works equally well. Always using one of them is a
known cause of long-term imbalance in delete-heavy workloads; alternating helps, and NB-08's
balanced trees make it moot.

In [3]:
# ---------------------------------------------------------------------------
# 1.2 The BST. Every operation iterative -- see Part 0.
# ---------------------------------------------------------------------------
class BST:
    """A binary search tree with distinct keys. No balancing -- that is Part 3's subject."""

    def __init__(self, keys=()):
        self.root = None
        self._size = 0
        for k in keys:
            self.insert(k)

    def __len__(self):
        return self._size

    def search(self, key):
        node = self.root
        while node is not None:
            if key == node.key:
                return True
            node = node.left if key < node.key else node.right
        return False

    def search_cost(self, key):
        """Comparisons performed. Part 3 counts these."""
        node, comparisons = self.root, 0
        while node is not None:
            comparisons += 1
            if key == node.key:
                return comparisons
            node = node.left if key < node.key else node.right
        return comparisons

    def insert(self, key):
        """Walk down as if searching; the slot you fall off is where it belongs."""
        if self.root is None:
            self.root = BNode(key)
            self._size += 1
            return True
        node = self.root
        while True:
            if key == node.key:
                return False                       # duplicates rejected
            if key < node.key:
                if node.left is None:
                    node.left = BNode(key)
                    self._size += 1
                    return True
                node = node.left
            else:
                if node.right is None:
                    node.right = BNode(key)
                    self._size += 1
                    return True
                node = node.right

    def delete(self, key):
        parent, node = None, self.root
        while node is not None and node.key != key:
            parent = node
            node = node.left if key < node.key else node.right
        if node is None:
            return False

        if node.left is not None and node.right is not None:
            # CASE 3: two children. Copy the inorder successor's key up, then
            # remove the successor -- which has no left child, so it is case 1 or 2.
            succ_parent, succ = node, node.right
            while succ.left is not None:
                succ_parent, succ = succ, succ.left
            node.key = succ.key
            if succ_parent.left is succ:
                succ_parent.left = succ.right
            else:
                succ_parent.right = succ.right
        else:
            # CASE 1 (no children) and CASE 2 (one child) are the same splice.
            child = node.left if node.left is not None else node.right
            if parent is None:
                self.root = child
            elif parent.left is node:
                parent.left = child
            else:
                parent.right = child

        self._size -= 1
        return True

    def inorder(self):
        """Iterative inorder -- NB-06 section 1.3."""
        out, stack, cur = [], [], self.root
        while stack or cur is not None:
            while cur is not None:
                stack.append(cur)
                cur = cur.left
            cur = stack.pop()
            out.append(cur.key)
            cur = cur.right
        return out

    def height(self):
        peak, stack = -1, [(self.root, 0)]
        while stack:
            node, d = stack.pop()
            if node is None:
                continue
            peak = max(peak, d)
            stack.append((node.left, d + 1))
            stack.append((node.right, d + 1))
        return peak


def bst_ok(tree):
    """The invariant, checked three independent ways."""
    keys = tree.inorder()
    if keys != sorted(keys):
        return "inorder is not sorted: %r" % (keys,)
    if len(set(keys)) != len(keys):
        return "duplicate keys present: %r" % (keys,)
    if len(keys) != tree._size:
        return "size says %d but %d keys are reachable" % (tree._size, len(keys))
    if not in_range(tree.root):
        return "the interval invariant is violated"
    return True


t = BST([50, 30, 70, 20, 40, 60, 80])
print("inserted 50, 30, 70, 20, 40, 60, 80")
print("  inorder:", t.inorder(), " height:", t.height(), " size:", len(t))
print()
print("  delete 20 (a leaf)          ->", t.delete(20), t.inorder())
print("  delete 30 (one child, 40)   ->", t.delete(30), t.inorder())
print("  delete 50 (TWO children)    ->", t.delete(50), t.inorder())
print()
print("  Deleting the root 50 replaced its key with 60, the smallest key in")
print("  its right subtree, then removed that 60 from where it was.")

inserted 50, 30, 70, 20, 40, 60, 80
  inorder: [20, 30, 40, 50, 60, 70, 80]  height: 2  size: 7

  delete 20 (a leaf)          -> True [30, 40, 50, 60, 70, 80]
  delete 30 (one child, 40)   -> True [40, 50, 60, 70, 80]
  delete 50 (TWO children)    -> True [40, 60, 70, 80]

  Deleting the root 50 replaced its key with 60, the smallest key in
  its right subtree, then removed that 60 from where it was.


In [4]:
# ---------------------------------------------------------------------------
# Differential test: the BST against a sorted set, invariant after every step.
# ---------------------------------------------------------------------------
def gen_ops(rng):
    """Small key range so deletes actually hit, and all three cases occur."""
    return [(rng.choice(["insert", "insert", "delete", "search"]), rng.randrange(-15, 15))
            for _ in range(rng.randrange(0, 50))]


def replay(ops):
    tree, ref = BST(), set()
    for op, key in ops:
        if op == "insert":
            got, want = tree.insert(key), key not in ref
            assert got == want, "insert(%r) returned %r, expected %r" % (key, got, want)
            ref.add(key)
        elif op == "delete":
            got, want = tree.delete(key), key in ref
            assert got == want, "delete(%r) returned %r, expected %r" % (key, got, want)
            ref.discard(key)
        else:
            assert tree.search(key) == (key in ref), "search(%r)" % key
        check_invariant(tree, bst_ok, "BST invariant", "%s %r" % (op, key))
        assert len(tree) == len(ref), "size %d vs %d" % (len(tree), len(ref))
    return tree.inorder()


def reference(ops):
    keys = set()
    for op, key in ops:
        if op == "insert":
            keys.add(key)
        elif op == "delete":
            keys.discard(key)
    return sorted(keys)


checked = stress(replay, reference, gen_ops, n=5000, seed=RANDOM_SEED, label="BST")
print("BST: %s randomised operation sequences agree with a sorted set --" % "{:,}".format(checked))
print("     return values included, with the invariant checked after every")
print("     single operation and all three delete cases exercised.")

BST: 5,000 randomised operation sequences agree with a sorted set --
     return values included, with the invariant checked after every
     single operation and all three delete cases exercised.


## 1.3 Inorder yields sorted order — the invariant restated

This is usually presented as a pleasing extra property. It is not extra: **it is the invariant,
stated differently.**

Inorder visits left subtree, then node, then right subtree. The invariant says everything in the
left subtree is smaller than the node and everything in the right is larger. So by induction the
output is sorted — and conversely, if the inorder output is sorted, the invariant holds. They are
the same statement, which is why §2.1 can validate a BST by checking sortedness.

**What this buys, and it is exactly what NB-03's hash table cannot do at any price:**

| Operation | BST | Hash table |
|---|---|---|
| search / insert / delete | $O(h)$ | $O(1)$ expected |
| **sorted iteration** | $\Theta(n)$, no sort needed | $\Theta(n \log n)$ — you must sort |
| **min / max** | $O(h)$ — walk to an end | $\Theta(n)$ — scan everything |
| **predecessor / successor** | $O(h)$ (§2.2) | $\Theta(n)$ |
| **range query** `[lo, hi]` | $O(h + k)$ (§2.4) | $\Theta(n)$ |

The hash table wins the first row and loses every other one. That is the whole trade, and it is why
both exist: **hashing destroys order to gain speed; a search tree preserves order and pays for it.**

In [5]:
# ---------------------------------------------------------------------------
# 1.3 Inorder is sorted -- and it is the invariant, not a bonus.
# ---------------------------------------------------------------------------
rng = random.Random(RANDOM_SEED)
keys = rng.sample(range(1000), 40)
tree = BST(keys)

print("40 keys inserted in random order.")
print("  first 12 inserted :", keys[:12])
print("  inorder (first 12):", tree.inorder()[:12])
print("  inorder == sorted :", tree.inorder() == sorted(keys))
print()

checked = stress(lambda ks: BST(ks).inorder(),
                 lambda ks: sorted(set(ks)),
                 lambda r: [r.randrange(-40, 40) for _ in range(r.randrange(0, 30))],
                 n=4000, seed=RANDOM_SEED, label="inorder sorted")
print("%s random key sets: inorder output is exactly the sorted distinct keys."
      % "{:,}".format(checked))

print()
print("And the operations the ordering buys, which a hash table cannot do:")
t = BST(rng.sample(range(1000), 200))
ks = t.inorder()
print("  min          ", ks[0], "   max", ks[-1], "  (each an O(h) walk to one end)")
lo, hi = 300, 340
print("  range [%d, %d] " % (lo, hi), [k for k in ks if lo <= k <= hi])
print("  a set() could answer none of these without scanning or sorting.")

40 keys inserted in random order.
  first 12 inserted : [426, 750, 10, 839, 845, 822, 305, 875, 377, 954, 198, 276]
  inorder (first 12): [10, 79, 93, 127, 151, 165, 170, 178, 191, 198, 210, 267]
  inorder == sorted : True

4,000 random key sets: inorder output is exactly the sorted distinct keys.

And the operations the ordering buys, which a hash table cannot do:
  min           3    max 998   (each an O(h) walk to one end)
  range [300, 340]  [308, 329, 331, 332, 333, 337, 338]
  a set() could answer none of these without scanning or sorting.


## 1.4 Java: `Comparable`, `Comparator`, and the contract that breaks `TreeSet`

A BST needs to order keys, and Java offers two ways to say how:

- **`Comparable<T>`** — the type's *natural* order, via `compareTo`. One per type.
- **`Comparator<T>`** — an external ordering passed to the container. As many as you like.

`TreeMap` and `TreeSet` are Java's ordered containers, and they are **red-black trees** (NB-08), so
they never degenerate the way §3's tree does. That is the practical answer to "why does `TreeMap`
exist when `HashMap` is faster?" — it exists for the second column of §1.3's table:
`firstKey`, `lastKey`, `headMap`, `tailMap`, `subMap`, `floorKey`, `ceilingKey`, and ordered
iteration. All of them are $O(\log n)$ on a tree and impossible on a hash map.

**And now the contract**, which is where this gets sharp. `compareTo`/`compare` is required to be
**consistent with `equals`**: `a.compareTo(b) == 0` should hold exactly when `a.equals(b)`. It is
only a *should* — nothing enforces it — and an ordered container that decides equality by
comparison rather than by `equals` will do something surprising when you break it.

The cell below breaks it deliberately.

In [6]:
# ---------------------------------------------------------------------------
# 1.4 What TreeSet does with a comparator inconsistent with equals.
# ---------------------------------------------------------------------------
JAVA_ORDER_SRC = r"""
import java.util.*;

public class Ordering {
    static final class Node { int key; Node left, right; Node(int k) { key = k; } }

    static int searchCost(Node n, int k) {
        int c = 0;
        while (n != null) {
            c++;
            if (k == n.key) return c;
            n = (k < n.key) ? n.left : n.right;
        }
        return c;
    }

    public static void main(String[] args) {
        final int n = 20_000;

        // A hand-rolled BST fed SORTED keys is a right spine. Built iteratively,
        // because building it recursively would overflow the stack (NB-06 section 3).
        Node root = null, tail = null;
        for (int i = 0; i < n; i++) {
            Node x = new Node(i);
            if (root == null) { root = x; } else { tail.right = x; }
            tail = x;
        }
        TreeMap<Integer,Integer> tm = new TreeMap<>();
        for (int i = 0; i < n; i++) tm.put(i, i);

        Random rnd = new Random(1);
        long cost = 0;
        for (int i = 0; i < 200; i++) cost += searchCost(root, rnd.nextInt(n));
        System.out.println("A. Sorted insertion of " + String.format("%,d", n) + " keys:");
        System.out.printf("   hand-rolled BST : %,d comparisons per search (mean of 200)%n",
                          cost / 200);
        System.out.printf("   TreeMap         : about %d, and it stays there --%n",
                          (int) (Math.log(n) / Math.log(2)) + 1);
        System.out.println("   TreeMap is a RED-BLACK tree; it rebalances on insert (NB-08).");
        System.out.println("   That is what TreeMap is FOR. HashMap is faster and unordered;");
        System.out.println("   TreeMap gives firstKey, floorKey, subMap and sorted iteration.");

        System.out.println();
        System.out.println("B. A comparator inconsistent with equals:");
        TreeSet<String> byLength = new TreeSet<>(Comparator.comparingInt(String::length));
        byLength.add("aaa");
        byLength.add("bbb");
        byLength.add("cccc");
        System.out.println("   added \"aaa\", \"bbb\", \"cccc\", ordered by LENGTH only");
        System.out.println("   contents        : " + byLength);
        System.out.println("   size            : " + byLength.size()
                           + "        <- \"bbb\" was silently dropped");
        System.out.println("   contains(\"bbb\") : " + byLength.contains("bbb")
                           + "     <- TRUE, for a string that is not in the set");
        System.out.println();
        System.out.println("   A TreeSet decides equality with compare() == 0, never with");
        System.out.println("   equals(). Two different strings of equal length are therefore");
        System.out.println("   the SAME element to it. A HashSet with the same three strings");
        System.out.println("   would hold all three.");

        System.out.println();
        System.out.println("C. Comparable (one natural order) vs Comparator (as many as you like):");
        List<String> words = new ArrayList<>(List.of("pear", "fig", "banana", "kiwi"));
        Collections.sort(words);
        System.out.println("   natural order              : " + words);
        words.sort(Comparator.comparingInt(String::length)
                             .thenComparing(Comparator.naturalOrder()));
        System.out.println("   by length, then alphabetical: " + words);
    }
}
"""

if JAVA:
    print(run_java(JAVA_ORDER_SRC, timeout=600))
else:
    print("JDK not available; skipping the Java section.")

A. Sorted insertion of 20,000 keys:
   hand-rolled BST : 9,469 comparisons per search (mean of 200)
   TreeMap         : about 15, and it stays there --
   TreeMap is a RED-BLACK tree; it rebalances on insert (NB-08).
   That is what TreeMap is FOR. HashMap is faster and unordered;
   TreeMap gives firstKey, floorKey, subMap and sorted iteration.

B. A comparator inconsistent with equals:
   added "aaa", "bbb", "cccc", ordered by LENGTH only
   contents        : [aaa, cccc]
   size            : 2        <- "bbb" was silently dropped
   contains("bbb") : true     <- TRUE, for a string that is not in the set

   A TreeSet decides equality with compare() == 0, never with
   equals(). Two different strings of equal length are therefore
   the SAME element to it. A HashSet with the same three strings
   would hold all three.

C. Comparable (one natural order) vs Comparator (as many as you like):
   natural order              : [banana, fig, kiwi, pear]
   by length, then alphabetical: [fig,

**Part B is the one to remember.** Three distinct strings go in; two come out; and
`contains("bbb")` returns **`true`** for a string the set does not contain.

Nothing is broken in `TreeSet`. It is doing exactly what an ordered container must: it locates
elements by *comparison*, because that is the only tool it has, so `compare(a, b) == 0` is its
definition of "same element". A comparator that calls `"aaa"` and `"bbb"` equal has told it they
are the same element, and it believed us.

**The practical rules:**

- **A comparator for an ordered container must be a total order consistent with `equals`.**
  If you want to sort by length, break ties on something — `thenComparing(naturalOrder())` — so
  distinct elements never compare equal. Part C shows the fix.
- **Comparators for *sorting* are held to a lower standard than comparators for *containers*.**
  `list.sort(byLength)` is fine: it may order equal-length strings arbitrarily, but it loses
  nothing. The same comparator in a `TreeSet` destroys data.
- **This is NB-03 §1.6's `hashCode`/`equals` contract with a different pair of methods**, and the
  same shape of consequence: violate the agreement between "how do I locate it" and "is it the
  same thing", and the container misfiles or loses entries silently. There, `equals` without
  `hashCode` produced two entries for one key; here, a coarse comparator produces one entry for two
  keys. Both compile, neither throws.

Part A is the answer to "why does `TreeMap` exist?" — **not** speed, which `HashMap` wins outright,
but §1.3's second column. And note that `TreeMap` shrugs off the sorted insertion that costs the
hand-rolled tree ~9,500 comparisons per search, because it rebalances. That gap is Part 3's subject
and NB-08's reason for existing.

***
# Part 2 - Worked problems

Four problems, each of which is the BST invariant used for something other than search.

| # | Problem | What it exploits |
|---|---|---|
| 2.1 | Validate a BST | the invariant is about **subtrees**, not children |
| 2.2 | Successor and predecessor | the last left turn you took |
| 2.3 | Lowest common ancestor | the split point is where the paths diverge |
| 2.4 | Range query and $k$th smallest | pruning whole subtrees by their bounds |

## 2.1 Validating a BST — and the check that looks right

§1.1 built the four-node tree that satisfies "every node is ordered against its children" while
being a broken BST. This section writes both validators and puts them in front of a differential
test.

**The wrong one** compares each node with its immediate children. It is the direct translation of
the wrong definition, it is the most common answer given under pressure, and it accepts §1.1's
tree.

**The right one** carries the interval down: each node must lie in $(lo, hi)$, and descending left
tightens $hi$ while descending right tightens $lo$. Every node is then checked against *all* its
ancestors, which is what the invariant actually demands.

**A third correct one** checks that the inorder traversal is strictly increasing — §1.3's
observation used as an algorithm. It is the shortest to write and the easiest to argue is right,
and it needs $\Theta(n)$ space unless you keep only the previous key.

The interesting part is not that the wrong version is wrong. It is **how quickly a randomised test
finds it**, given that hand-picked examples so often miss it.

In [7]:
# ---------------------------------------------------------------------------
# 2.1 Three validators: one wrong, two right.
# ---------------------------------------------------------------------------
def is_bst_children_only(root):
    """WRONG: compares each node only with its immediate children."""
    stack = [root]
    while stack:
        n = stack.pop()
        if n is None:
            continue
        if n.left is not None and n.left.key >= n.key:
            return False
        if n.right is not None and n.right.key <= n.key:
            return False
        stack.append(n.left)
        stack.append(n.right)
    return True


def is_bst_intervals(root):
    """RIGHT: every node must lie inside the interval its ancestors imply."""
    stack = [(root, None, None)]
    while stack:
        n, lo, hi = stack.pop()
        if n is None:
            continue
        if lo is not None and n.key <= lo:
            return False
        if hi is not None and n.key >= hi:
            return False
        stack.append((n.left, lo, n.key))
        stack.append((n.right, n.key, hi))
    return True


def is_bst_inorder(root):
    """RIGHT: the inorder traversal must be strictly increasing (section 1.3)."""
    stack, cur, prev = [], root, None
    while stack or cur is not None:
        while cur is not None:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        if prev is not None and cur.key <= prev:
            return False
        prev = cur.key
        cur = cur.right
    return True


print("On section 1.1's four-node tree (10; left 5; right 15; 15's left child 6):")
print("  is_bst_children_only ->", is_bst_children_only(broken), " <- WRONG")
print("  is_bst_intervals     ->", is_bst_intervals(broken))
print("  is_bst_inorder       ->", is_bst_inorder(broken))


def build_arbitrary(keys):
    """Build a tree of the given keys in level order, ignoring ordering entirely.

    Most results are NOT valid BSTs, which is exactly what we want to test with.
    """
    if not keys:
        return None
    nodes = [BNode(k) for k in keys]
    for i, node in enumerate(nodes):
        left, right = 2 * i + 1, 2 * i + 2
        node.left = nodes[left] if left < len(nodes) else None
        node.right = nodes[right] if right < len(nodes) else None
    return nodes[0]


def gen_keys(rng):
    n = rng.randrange(0, 9)
    return rng.sample(range(20), n)


print()
print("Now hand the wrong validator to a differential test against the right one,")
print("on randomly shaped trees of distinct keys:")
print()
try:
    n = stress(lambda ks: is_bst_children_only(build_arbitrary(ks)),
               lambda ks: is_bst_intervals(build_arbitrary(ks)),
               gen_keys, n=4000, seed=RANDOM_SEED, label="is_bst_children_only")
    print("  no disagreement in %d cases -- the test is not strong enough" % n)
except StressFailure as exc:
    for line in str(exc).split("\n"):
        print("  " + line)

On section 1.1's four-node tree (10; left 5; right 15; 15's left child 6):
  is_bst_children_only -> True  <- WRONG
  is_bst_intervals     -> False
  is_bst_inorder       -> False

Now hand the wrong validator to a differential test against the right one,
on randomly shaped trees of distinct keys:

  is_bst_children_only: implementation disagrees with reference
    failing input : [13, 12, 15, 2, 18]
    implementation: True
    reference     : False


In [8]:
# ---------------------------------------------------------------------------
# And the two correct validators agree with each other, always.
# ---------------------------------------------------------------------------
checked = stress(lambda ks: is_bst_intervals(build_arbitrary(ks)),
                 lambda ks: is_bst_inorder(build_arbitrary(ks)),
                 gen_keys, n=4000, seed=RANDOM_SEED, label="intervals vs inorder")
print("intervals vs inorder: %s randomly shaped trees, identical verdicts."
      % "{:,}".format(checked))

checked = stress(lambda ks: is_bst_intervals(BST(ks).root),
                 lambda ks: True,
                 gen_keys, n=4000, seed=RANDOM_SEED, label="BST is a BST")
print("BST is a BST:         %s trees built by insert() all validate."
      % "{:,}".format(checked))

intervals vs inorder: 4,000 randomly shaped trees, identical verdicts.
BST is a BST:         4,000 trees built by insert() all validate.


**The randomised test found a counterexample immediately** — a small tree the wrong validator calls
a BST and the interval validator rejects. Note what it took: no cleverness, no hand-picked
adversarial case, just randomly shaped trees and a correct reference.

That is the general lesson, and it is the same one as NB-05 §3.1: **a plausible-looking check
needs a reference implementation, not more staring.** The wrong validator is not obviously wrong —
it passes every balanced example, every sorted example, and most random ones. It fails only when a
key is misplaced across an ancestor's boundary rather than a parent's, which is a case people do
not think to draw.

Two details worth carrying:

- **`is_bst_inorder` must compare against the previous key, not collect a list and sort it.**
  Sorting the collected keys and comparing tells you the multiset is sorted, which is vacuously
  true. Compare adjacent keys as you go, and use `<=` to reject duplicates if your convention
  rejects them (§1.1).
- **The interval version needs open bounds and `None` for infinity.** Using `float("-inf")` works
  for numbers and breaks for strings or tuples; `None`-means-unbounded works for anything ordered.

## 2.2 Successor and predecessor

**The problem.** Given a key, find the next larger key in the tree — its **inorder successor**.

Without parent pointers there is a neat one-pass answer, and the reasoning is worth following
because it is the invariant used in an unusual direction.

Walk down as if searching. At each node:

- if the node's key is **greater** than the target, this node is a *candidate* successor — record
  it and go left, hoping for something smaller but still greater;
- if the node's key is $\le$ the target, nothing here or to its left can be the successor, so go
  right.

The answer is **the last candidate recorded**, which is to say **the node at the last left turn**.
No parent pointers, no second pass, $O(h)$.

The predecessor is the mirror image: record on going right, take the last right turn.

**Why this matters beyond the exercise:** successor is what makes an in-order *iterator* possible
without a stack, it is `TreeMap.higherKey` and `ceilingKey`, and §1.2's delete used the successor
in its two-child case — there, though, the successor was known to be in the right subtree, which is
the easy case of this problem.

In [9]:
# ---------------------------------------------------------------------------
# 2.2 Successor and predecessor: the last turn you took.
# ---------------------------------------------------------------------------
def successor(root, target):
    """Smallest key strictly greater than target, or None."""
    best, node = None, root
    while node is not None:
        if node.key > target:
            best = node.key          # candidate: remember it, look for a smaller one
            node = node.left
        else:
            node = node.right        # too small: the successor must be to the right
    return best


def predecessor(root, target):
    """Largest key strictly less than target, or None."""
    best, node = None, root
    while node is not None:
        if node.key < target:
            best = node.key
            node = node.right
        else:
            node = node.left
    return best


def gen_query(rng):
    keys = rng.sample(range(40), rng.randrange(0, 20))
    return (keys, rng.randrange(-5, 45))


def succ_ref(case):
    keys, target = case
    larger = sorted(k for k in set(keys) if k > target)
    return larger[0] if larger else None


def pred_ref(case):
    keys, target = case
    smaller = sorted(k for k in set(keys) if k < target)
    return smaller[-1] if smaller else None


for name, impl, ref in (("successor", successor, succ_ref),
                        ("predecessor", predecessor, pred_ref)):
    checked = stress(lambda c, f=impl: f(BST(c[0]).root, c[1]),
                     lambda c, g=ref: g(c),
                     gen_query, n=4000, seed=RANDOM_SEED, label=name)
    print("%-13s %s random (tree, target) pairs agree, including targets that are"
          % (name + ":", "{:,}".format(checked)))
    print("%-13s absent, below the minimum and above the maximum." % "")

t = BST([50, 30, 70, 20, 40, 60, 80])
print()
print("  tree inorder:", t.inorder())
for target in (40, 45, 80, 10):
    print("    target %-3d -> predecessor %-6s successor %s"
          % (target, predecessor(t.root, target), successor(t.root, target)))

successor:    4,000 random (tree, target) pairs agree, including targets that are
              absent, below the minimum and above the maximum.


predecessor:  4,000 random (tree, target) pairs agree, including targets that are
              absent, below the minimum and above the maximum.

  tree inorder: [20, 30, 40, 50, 60, 70, 80]
    target 40  -> predecessor 30     successor 50
    target 45  -> predecessor 40     successor 50
    target 80  -> predecessor 70     successor None
    target 10  -> predecessor None   successor 20


## 2.3 Lowest common ancestor — where the paths diverge

NB-06 Practice 4 does LCA on a **plain** binary tree: a postorder pass returning "did I find one of
them?", $\Theta(n)$, and fiddly. On a BST it is $O(h)$ and almost trivial, which is a clean
demonstration of what the invariant buys.

**The observation.** The paths from the root to $p$ and to $q$ start together and diverge exactly
once. The node where they diverge is the LCA. And the invariant tells you where that is without
walking either path:

- if **both** keys are less than the current node, both paths go left — descend left;
- if **both** are greater, descend right;
- otherwise the keys **split** here (one on each side, or one *is* this node) — this node is the
  LCA.

$O(h)$, no recursion, no second pass, and no need for the keys to be present as long as you state
what you mean when they are not.

In [10]:
# ---------------------------------------------------------------------------
# 2.3 LCA in a BST: descend until the keys split.
# ---------------------------------------------------------------------------
def lca(root, p, q):
    """Lowest common ancestor of keys p and q. O(h)."""
    lo, hi = (p, q) if p <= q else (q, p)
    node = root
    while node is not None:
        if hi < node.key:
            node = node.left         # both smaller
        elif lo > node.key:
            node = node.right        # both larger
        else:
            return node.key          # they split here -- this is the LCA
    return None


def lca_ref(case):
    """Reference: build both root-to-key paths and take the last shared node."""
    keys, p, q = case
    tree = BST(keys)

    def path_to(target):
        out, node = [], tree.root
        while node is not None:
            out.append(node.key)
            if target == node.key:
                return out
            node = node.left if target < node.key else node.right
        return out          # target absent: the path we would have taken

    a, b = path_to(p), path_to(q)
    shared = None
    for x, y in zip(a, b):
        if x != y:
            break
        shared = x
    return shared


def gen_lca(rng):
    keys = rng.sample(range(40), rng.randrange(1, 20))
    return (keys, rng.choice(keys), rng.choice(keys))


checked = stress(lambda c: lca(BST(c[0]).root, c[1], c[2]), lca_ref,
                 gen_lca, n=4000, seed=RANDOM_SEED, label="lca")
print("lca: %s random (tree, p, q) triples agree with a build-both-paths"
      % "{:,}".format(checked))
print("     reference, including p == q and one key being the other's ancestor.")

t = BST([50, 30, 70, 20, 40, 60, 80])
print()
print("  tree inorder:", t.inorder())
for p, q in ((20, 40), (20, 80), (60, 80), (30, 30)):
    print("    lca(%2d, %2d) = %s" % (p, q, lca(t.root, p, q)))
print()
print("  NB-06's plain-tree LCA is Theta(n) and needs a postorder pass.")
print("  Here it is O(h) and needs three comparisons per level. That gap is")
print("  what the ordering invariant is worth.")

lca: 4,000 random (tree, p, q) triples agree with a build-both-paths
     reference, including p == q and one key being the other's ancestor.

  tree inorder: [20, 30, 40, 50, 60, 70, 80]
    lca(20, 40) = 30
    lca(20, 80) = 50
    lca(60, 80) = 70
    lca(30, 30) = 30

  NB-06's plain-tree LCA is Theta(n) and needs a postorder pass.
  Here it is O(h) and needs three comparisons per level. That gap is
  what the ordering invariant is worth.


## 2.4 Range queries and $k$th smallest — pruning by bounds

**Range query.** Report every key in $[lo, hi]$. The naive approach walks the whole tree and
filters — $\Theta(n)$, and it throws the invariant away. The right approach **prunes**: at a node
with key $x$, if $x \le lo$ then nothing in the left subtree can be in range, so do not go there;
symmetrically on the right.

The cost is $O(h + k)$ for $k$ reported keys, and the measurement below counts node visits to
confirm the pruning is real rather than decorative.

**$k$th smallest.** An inorder walk stopped after $k$ elements is $O(h + k)$ — fine when $k$ is
small, $\Theta(n)$ when $k$ is near $n$. The proper answer is to **augment the tree**: store in each
node the size of its subtree, and then $k$th-smallest is a descent, $O(h)$:

- if `size(left) == k - 1`, this node is the answer;
- if `size(left) >= k`, recurse left;
- otherwise recurse right, looking for the $(k - size(left) - 1)$th.

That augmentation is worth noticing as a technique in its own right: **store a summary of each
subtree in its root, maintain it on every update, and queries that would have been $\Theta(n)$
become $O(h)$.** It is the same idea as NB-12's Fenwick and segment trees, and it is how
order-statistic trees and interval trees are built.

In [11]:
# ---------------------------------------------------------------------------
# 2.4 Range query with pruning, and kth smallest two ways.
# ---------------------------------------------------------------------------
def range_query(root, lo, hi):
    """Every key in [lo, hi], in sorted order, without visiting the whole tree."""
    out, stack, cur = [], [], root
    while stack or cur is not None:
        while cur is not None:
            stack.append(cur)
            cur = cur.left if cur.key > lo else None    # prune: left is all <= lo
        cur = stack.pop()
        if lo <= cur.key <= hi:
            out.append(cur.key)
        cur = cur.right if cur.key < hi else None       # prune: right is all >= hi
    return out


def range_query_visits(root, lo, hi):
    """The same walk, counting how many nodes it touches."""
    visits, stack, cur = 0, [], root
    while stack or cur is not None:
        while cur is not None:
            visits += 1
            stack.append(cur)
            cur = cur.left if cur.key > lo else None
        cur = stack.pop()
        cur = cur.right if cur.key < hi else None
    return visits


def kth_smallest_walk(root, k):
    """Inorder, stopped early. O(h + k)."""
    stack, cur, seen = [], root, 0
    while stack or cur is not None:
        while cur is not None:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        seen += 1
        if seen == k:
            return cur.key
        cur = cur.right
    return None


def gen_range(rng):
    keys = rng.sample(range(60), rng.randrange(0, 30))
    a, b = rng.randrange(-5, 65), rng.randrange(-5, 65)
    return (keys, min(a, b), max(a, b))


checked = stress(lambda c: range_query(BST(c[0]).root, c[1], c[2]),
                 lambda c: sorted(k for k in set(c[0]) if c[1] <= k <= c[2]),
                 gen_range, n=4000, seed=RANDOM_SEED, label="range_query")
print("range_query: %s random (tree, lo, hi) agree with a filtered sort."
      % "{:,}".format(checked))

checked = stress(lambda c: kth_smallest_walk(BST(c[0]).root, c[1]),
                 lambda c: (lambda s: s[c[1] - 1] if 0 < c[1] <= len(s) else None)(
                     sorted(set(c[0]))),
                 lambda r: (r.sample(range(60), r.randrange(0, 25)), r.randrange(1, 26)),
                 n=4000, seed=RANDOM_SEED, label="kth_smallest")
print("kth_smallest: %s random (tree, k) agree, including k beyond the size."
      % "{:,}".format(checked))

print()
print("Does the pruning actually prune? Nodes visited for a range query")
print("on a randomly built tree of 20,000 keys:")
print()
big = BST(random.Random(RANDOM_SEED).sample(range(200_000), 20_000))
print("  %-26s %12s %10s" % ("range", "keys found", "nodes visited"))
print("  " + "-" * 52)
for lo, hi in ((100_000, 100_100), (100_000, 101_000), (100_000, 110_000), (0, 200_000)):
    found = range_query(big.root, lo, hi)
    visits = range_query_visits(big.root, lo, hi)
    print("  %-26s %12s %10s"
          % ("[%s, %s]" % ("{:,}".format(lo), "{:,}".format(hi)),
             "{:,}".format(len(found)), "{:,}".format(visits)))
print()
print("  A narrow range touches a few dozen nodes out of 20,000. The whole-tree")
print("  range touches all of them, as it must. That is O(h + k), not O(n).")

range_query: 4,000 random (tree, lo, hi) agree with a filtered sort.
kth_smallest: 4,000 random (tree, k) agree, including k beyond the size.

Does the pruning actually prune? Nodes visited for a range query
on a randomly built tree of 20,000 keys:



  range                        keys found nodes visited
  ----------------------------------------------------
  [100,000, 100,100]                    8         25
  [100,000, 101,000]                  100        115
  [100,000, 110,000]                1,004      1,020
  [0, 200,000]                     20,000     20,000

  A narrow range touches a few dozen nodes out of 20,000. The whole-tree
  range touches all of them, as it must. That is O(h + k), not O(n).


***
# Part 3 - The signature difficulty: nothing enforces balance

Every cost in this notebook is $O(h)$. Search, insert, delete, successor, LCA, range query — all of
them $O(h)$, and $O(h)$ sounds like $O(\log n)$ because that is what a tree "obviously" is.

**Nothing in the BST definition bounds $h$.** The invariant constrains *order*; it says nothing
about *shape*. `insert` walks down and attaches a leaf, never restructuring anything — which is
what makes it simple, and what lets the tree become whatever the insertion order dictates.

And the worst insertion order is not an exotic adversarial construction. **It is sorted input**,
which is the most common way data arrives: read from a sorted file, drawn from an ordered database
query, generated by a counter, timestamps, auto-increment IDs. Every key is larger than the last, so
every key goes right, and the tree is a linked list.

Three measurements: the **shape** (§3.1), what it **costs** (§3.2), and what to do (§3.3).

## 3.1 The shape, by insertion order

The same 50,000 keys, inserted three ways:

- **sorted** — the pathological case, and the common one;
- **random** — the realistic average case;
- **"balanced"** — inserting in the order a perfectly balanced tree would need (median first, then
  recursively the medians of each half), which is the best achievable.

The theory to check against, for random insertion:

- **average node depth** $\approx 2 \ln n - 3$, so a *typical* search is about $1.39 \log_2 n$
  comparisons — around 39% worse than optimal;
- **expected height** $\approx 4.311 \ln n$ asymptotically, which is about $2.99 \log_2 n$.

The second one is worth measuring rather than quoting, because the convergence is slow.

In [12]:
# ---------------------------------------------------------------------------
# 3.1 Height and average depth, by insertion order.
# ---------------------------------------------------------------------------
def keys_in_order(kind, n, seed=0):
    keys = list(range(n))
    if kind == "sorted":
        return keys
    if kind == "random":
        random.Random(seed).shuffle(keys)
        return keys
    out = []                                    # "balanced": medians first
    stack = [(0, n - 1)]
    while stack:
        lo, hi = stack.pop()
        if lo > hi:
            continue
        mid = (lo + hi) // 2
        out.append(mid)
        stack.append((lo, mid - 1))
        stack.append((mid + 1, hi))
    return out


def average_depth(tree):
    total, count, stack = 0, 0, [(tree.root, 0)]
    while stack:
        node, d = stack.pop()
        if node is None:
            continue
        total += d
        count += 1
        stack.append((node.left, d + 1))
        stack.append((node.right, d + 1))
    return total / count if count else 0


N = 50_000
print("The same %s keys, inserted three ways:" % "{:,}".format(N))
print()
print("  %-12s %12s %14s %14s" % ("order", "height", "avg depth", "log2(n)"))
print("  " + "-" * 56)
for kind in ("balanced", "random", "sorted"):
    tree = BST(keys_in_order(kind, N, seed=RANDOM_SEED))
    print("  %-12s %12s %14.2f %14.2f"
          % (kind, "{:,}".format(tree.height()), average_depth(tree), math.log2(N)))
    del tree

print()
print("Random insertion against its theory, averaged over independent trials:")
print()
print("  %8s %7s %12s %12s %12s %10s" % ("n", "trials", "avg depth", "2 ln n - 3",
                                         "height", "h / ln n"))
print("  " + "-" * 66)
for n, trials in ((1_000, 40), (10_000, 20), (100_000, 5)):
    heights, depths = [], []
    for s in range(trials):
        tree = BST(keys_in_order("random", n, seed=s * 7919 + n))
        heights.append(tree.height())
        depths.append(average_depth(tree))
        del tree
    ln = math.log(n)
    print("  %8s %7d %12.2f %12.2f %12.2f %10.3f"
          % ("{:,}".format(n), trials, statistics.mean(depths), 2 * ln - 3,
             statistics.mean(heights), statistics.mean(heights) / ln))

print()
print("  Average depth tracks 2 ln n - 3 closely.")
print("  The height's asymptotic constant is 4.311, and h/ln n is still only")
print("  around 3.4 at n = 100,000 -- it converges slowly, so quoting 4.311 ln n")
print("  as the height at these sizes overstates it by roughly a quarter.")

The same 50,000 keys, inserted three ways:

  order              height      avg depth        log2(n)
  --------------------------------------------------------


  balanced               15          13.69          15.61


  random                 34          18.05          15.61


  sorted             49,999       24999.50          15.61

Random insertion against its theory, averaged over independent trials:

         n  trials    avg depth   2 ln n - 3       height   h / ln n
  ------------------------------------------------------------------
     1,000      40        11.00        10.82        20.70      2.997


    10,000      20        15.77        15.42        30.45      3.306


   100,000       5        20.63        20.03        39.60      3.440

  Average depth tracks 2 ln n - 3 closely.
  The height's asymptotic constant is 4.311, and h/ln n is still only
  around 3.4 at n = 100,000 -- it converges slowly, so quoting 4.311 ln n
  as the height at these sizes overstates it by roughly a quarter.


## 3.2 What the shape costs

In [13]:
# ---------------------------------------------------------------------------
# 3.2 Comparisons per search, and the cost of building the tree.
# ---------------------------------------------------------------------------
print("Comparisons per successful search, averaged over random lookups:")
print()
print("  %10s %14s %14s %16s %12s" % ("n", "balanced", "random", "sorted", "log2(n)"))
print("  " + "-" * 72)
for n in (1_000, 10_000, 50_000):
    row = []
    for kind in ("balanced", "random", "sorted"):
        tree = BST(keys_in_order(kind, n, seed=RANDOM_SEED))
        rng = random.Random(99)
        probes = [rng.randrange(n) for _ in range(2_000 if kind != "sorted" else 200)]
        row.append(statistics.mean(tree.search_cost(k) for k in probes))
        del tree
    print("  %10s %14.1f %14.1f %16s %12.1f"
          % ("{:,}".format(n), row[0], row[1],
             "{:,.0f}".format(row[2]), math.log2(n)))

print()
print("  At n = 50,000 the sorted-insertion tree costs about %d times more"
      % round(row[2] / row[0]))
print("  comparisons per search than the balanced one. Same keys, same code,")
print("  same invariant -- only the order they arrived in.")

Comparisons per successful search, averaged over random lookups:

           n       balanced         random           sorted      log2(n)
  ------------------------------------------------------------------------
       1,000            9.0           12.3              480         10.0


      10,000           12.4           16.2            4,938         13.3


      50,000           14.7           19.1           24,726         15.6

  At n = 50,000 the sorted-insertion tree costs about 1679 times more
  comparisons per search than the balanced one. Same keys, same code,
  same invariant -- only the order they arrived in.


In [14]:
# ---------------------------------------------------------------------------
# And building the tree at all.
# ---------------------------------------------------------------------------
def build_sorted(n):
    tree = BST()
    for k in range(n):
        tree.insert(k)
    return n


def build_random(n):
    tree = BST()
    for k in keys_in_order("random", n, seed=n):
        tree.insert(k)
    return n


print("Building the tree -- n inserts:")
print()
print("sorted input:")
growth_table(measure_growth(build_sorted, [1_000, 2_000, 4_000, 8_000], repeats=3),
             claim="O(n^2)")
print()
print("random input:")
growth_table(measure_growth(build_random, [50_000, 100_000, 200_000, 400_000], repeats=3),
             claim="O(n log n)")

Building the tree -- n inserts:

sorted input:


         n        seconds      ratio
------------------------------------
     1,000       0.041185          -
     2,000       0.174287       4.23
     4,000       0.677298       3.89
     8,000       2.814718       4.16

best fit: O(n^2) (relative error 0.026); next: O(n log n) (1.192)
claimed O(n^2) -> measurement MATCHES the claim

random input:


         n        seconds      ratio
------------------------------------
    50,000       0.130064          -
   100,000       0.320723       2.47
   200,000       0.753146       2.35
   400,000       1.780190       2.36

best fit: O(n log n) (relative error 0.138); next: O(n) (0.212)
claimed O(n log n) -> measurement MATCHES the claim


[('O(n log n)', 0.13775401027780132),
 ('O(n)', 0.21181939682045475),
 ('O(n^2)', 0.854866321074863),
 ('O(log n)', 2.0920161219049707),
 ('O(1)', 2.476114901160979),
 ('O(2^n)', 2.476114901160979),
 ('O(n^3)', 6.346644904573261)]

**$\Theta(n^2)$ to build from sorted input against $\Theta(n \log n)$ from random**, and roughly
**1,700× more comparisons per search** at n = 50,000.

Three things worth separating, because they are usually run together:

- **Random insertion is fine.** Average depth about 39% above optimal, height about 2–3× optimal —
  a constant factor, not a complexity change. A BST fed genuinely random keys is a perfectly good
  data structure, and the measurements say so.
- **Sorted insertion is a catastrophe**, and it is the common case rather than the exotic one.
  Data arrives sorted constantly: exported in key order, produced by a counter, timestamps,
  auto-increment IDs, or simply because someone sorted it before loading it. Nearly-sorted input is
  nearly as bad.
- **The failure is silent.** Nothing throws. Every operation returns the right answer. The tree is
  a *valid* BST — §2.1's validators all pass — it is just $\Theta(n)$ per operation. You discover it
  when a service that was fine in testing with shuffled fixtures falls over on production data that
  happens to be ordered.

And once again: **who chooses the input?** If the insertion order can be influenced from outside,
sorted input is a denial-of-service vector requiring no cleverness at all — just send your keys in
order. That is the fourth structure in this series with the same shape of weakness, after NB-02
§3's string matching, NB-03 §3's hash flooding and NB-06 §3's tree depth.

## 3.3 What to do about it

**Four answers, in rough order of practicality.**

**1. Use a balanced tree (NB-08).** Red-black trees, AVL trees and B-trees add rebalancing to
`insert` and `delete` and guarantee $h = O(\log n)$ *whatever* the insertion order. This is what
Java's `TreeMap` does — §1.4 measured it shrugging off the sorted insertion that costs the
hand-rolled tree ~9,500 comparisons per search — and what C++'s `std::map` does. **It is the
answer**, and the entire reason NB-08 is the next notebook.

**2. Shuffle the input.** If you have all the keys up front and do not need them inserted in order,
randomising is one line and buys the "random" row of §3.1. This is a *treap*'s idea made manual;
a treap gets the same guarantee by assigning random priorities, without needing the keys in advance.

**3. Build it balanced directly.** If the keys are already sorted, do not insert them one by one —
recursively take the median as the root. That is the "balanced" row above, it is $\Theta(n)$, and
it produces a perfect tree. Practice 6 does this.

**4. Do you need a tree at all?** §1.3's table is the honest question. If you never do ordered
operations — no range queries, no successor, no sorted iteration, no min/max — then a **hash table**
(NB-03) gives $O(1)$ expected and cannot degenerate this way. Reach for a tree when you need order;
reach for a hash when you need speed.

The one thing not on the list is "hope the input is random", which is what an unbalanced BST
silently assumes.

In [15]:
# ---------------------------------------------------------------------------
# 3.3 Fixes 2 and 3, measured against the disease.
# ---------------------------------------------------------------------------
N = 20_000
sorted_keys = list(range(N))

print("Loading %s keys that arrive ALREADY SORTED:" % "{:,}".format(N))
print()
print("  %-42s %10s %14s" % ("strategy", "height", "cost/search"))
print("  " + "-" * 70)

rng = random.Random(99)
probes = [rng.randrange(N) for _ in range(1_000)]

# 1. naive
tree = BST(sorted_keys)
naive_cost = statistics.mean(tree.search_cost(k) for k in probes[:100])
print("  %-42s %10s %14s"
      % ("insert one by one (naive)", "{:,}".format(tree.height()),
         "{:,.0f}".format(naive_cost)))
del tree

# 2. shuffle first
shuffled = list(sorted_keys)
random.Random(RANDOM_SEED).shuffle(shuffled)
tree = BST(shuffled)
print("  %-42s %10s %14.1f"
      % ("shuffle, then insert", "{:,}".format(tree.height()),
         statistics.mean(tree.search_cost(k) for k in probes)))
del tree

# 3. build balanced from the sorted keys directly
tree = BST(keys_in_order("balanced", N))
print("  %-42s %10s %14.1f"
      % ("build balanced from the sorted keys", "{:,}".format(tree.height()),
         statistics.mean(tree.search_cost(k) for k in probes)))
del tree

print("  %-42s %10.0f %14s" % ("(theoretical optimum)", math.log2(N), "-"))
print()
print("Two one-line changes take the same data from a linked list to a tree.")
print("Neither helps if the keys arrive over time rather than all at once --")
print("which is why NB-08 exists.")

Loading 20,000 keys that arrive ALREADY SORTED:

  strategy                                       height    cost/search
  ----------------------------------------------------------------------


  insert one by one (naive)                      19,999          9,586
  shuffle, then insert                               34           18.8
  build balanced from the sorted keys                14           13.3
  (theoretical optimum)                              14              -

Two one-line changes take the same data from a linked list to a tree.
Neither helps if the keys arrive over time rather than all at once --
which is why NB-08 exists.


***
# Part 4 - Tough questions

***

### Q1. State the BST invariant. Precisely.

<details><summary>Answer</summary>

> For every node $x$: **every** key in $x$'s left subtree is less than $x$.key, and **every** key
> in $x$'s right subtree is greater than $x$.key.

**Not** "its left child is smaller and its right child is larger." That weaker statement is a
different property, and §1.1 builds the tree that satisfies it while being useless:

```
     10
    /  \
   5    15
       /
      6        <- in 10's RIGHT subtree, but smaller than 10
```

Every node is correctly ordered against its own children. Searching for `6` goes left at the root
(because $6 < 10$) and never looks in the right subtree where `6` actually is. §1.1 runs that
search and shows it failing.

**Two equivalent formulations**, both useful:

- **Intervals.** Each node is confined to an open interval $(lo, hi)$ inherited from its ancestors;
  going left tightens $hi$ to the parent's key, going right tightens $lo$. This is §2.1's correct
  validator.
- **Inorder is strictly increasing.** This is not a bonus property, it *is* the invariant restated
  (§1.3), which is why checking sortedness is a valid validator.

**And say what you do with duplicates** — reject, count per node, or send consistently one way with
one comparison weakened to $\le$. All three are defensible; mixing them gives a tree where a key is
present twice and only one copy is findable.

</details>

***

### Q2. Walk through deleting from a BST.

<details><summary>Answer</summary>

**Three cases**, and the third is the only interesting one.

1. **No children.** Detach it.
2. **One child.** Splice the child into its place. The child's subtree already satisfies whatever
   bound the deleted node satisfied, so the invariant is untouched.
3. **Two children.** You have two subtrees and one hole. **Replace the key with its inorder
   successor** — the smallest key in the right subtree — then delete *that* node from the right
   subtree.

**Why the successor is the only valid replacement:** it is greater than everything in the left
subtree (it is in the right one) and smaller than everything else in the right subtree (it is the
minimum of it). No other key satisfies both.

**Why the recursion terminates:** the successor is a *leftmost* node, so it has **no left child**,
so removing it is always case 1 or 2 — never case 3 again. There is exactly one level of
indirection, not an unbounded chain.

The **predecessor** (largest in the left subtree) works symmetrically. Always taking the successor
is a known source of long-term leftward imbalance in delete-heavy workloads; alternating helps, and
a balanced tree (NB-08) makes it irrelevant.

**Implementation note from §1.2:** cases 1 and 2 are the same code — splice in
`node.left if node.left else node.right`, which is `None` for a leaf. Writing them separately is
two branches where one will do.

</details>

***

### Q3. How do you validate that a tree is a BST?

<details><summary>Answer</summary>

**Two correct ways, and one that looks correct.**

**Wrong:** compare each node with its immediate children. This is the direct translation of Q1's
wrong definition and accepts §1.1's four-node tree.

**Right — intervals:** carry $(lo, hi)$ down; each node must lie strictly inside, with left
tightening $hi$ and right tightening $lo$. Every node is checked against *all* its ancestors.

**Right — inorder:** the traversal must be strictly increasing. Compare each key against the
**previous** one as you go; do not collect a list and sort it, which is vacuously true.

**The part worth reporting from §2.1:** handed to a differential test against the correct version
on randomly shaped trees, the wrong validator was caught **immediately** — a five-node
counterexample, no cleverness required. It passes every balanced example, every sorted example and
most random ones; it fails only when a key is misplaced across an *ancestor's* boundary rather than
a *parent's*, which is exactly the case people do not think to draw by hand.

That is the transferable lesson, and it is NB-05 §3.1's again: **a plausible check needs a
reference implementation, not more staring at it.**

**Detail:** use `None` for unbounded rather than `float("-inf")`. The infinity trick works for
numbers and breaks for strings, tuples, or anything else merely ordered.

</details>

***

### Q4. Why does inorder traversal give sorted output?

<details><summary>Answer</summary>

**Because that is what the invariant says**, restated. It is not a separate fact to remember.

Inorder visits left subtree, node, right subtree. The invariant says everything in the left subtree
is smaller than the node and everything in the right is larger. By induction the output is sorted;
and conversely, if the inorder output is sorted the invariant holds. The implication runs both
ways, which is why §2.1 can validate a BST by checking sortedness.

**What it buys — §1.3's table, and it is the whole reason to prefer a tree over a hash table:**

| Operation | BST | Hash table |
|---|---|---|
| search / insert / delete | $O(h)$ | $O(1)$ expected |
| **sorted iteration** | $\Theta(n)$, already sorted | $\Theta(n \log n)$ — must sort |
| **min / max** | $O(h)$ | $\Theta(n)$ |
| **predecessor / successor** | $O(h)$ | $\Theta(n)$ |
| **range query** | $O(h + k)$ | $\Theta(n)$ |

The hash table wins the first row outright and loses every other one. **Hashing destroys order to
gain speed; a search tree preserves order and pays for it.** That is the entire trade, and it is
why both exist — and why `HashMap` and `TreeMap` are both in the JDK.

</details>

***

### Q5. What is the BST's worst case, and how do you reach it?

<details><summary>Answer</summary>

**$\Theta(n)$ per operation, and you reach it by inserting sorted keys.**

Every cost in this notebook is $O(h)$, and **nothing in the definition bounds $h$.** The invariant
constrains *order*, not *shape*. `insert` walks down and attaches a leaf without restructuring
anything — that simplicity is exactly what allows the degeneration.

§3 measured the same 50,000 keys inserted three ways:

| Insertion order | Height | Comparisons per search |
|---|---|---|
| balanced (medians first) | 15 | **14.7** |
| random | 34 | **19.1** |
| **sorted** | **49,999** | **~24,700** |

Roughly **1,700× worse**, and building the tree is a measured $\Theta(n^2)$ against
$\Theta(n \log n)$ for random input.

**The reason this matters more than most worst cases:** sorted input is not exotic. Data arrives in
key order constantly — exported from a database, produced by a counter, timestamps, auto-increment
IDs, or simply sorted by someone before loading. Near-sorted is nearly as bad.

**And it fails silently.** Nothing throws, every answer is correct, and §2.1's validators all pass —
it is a perfectly valid BST that happens to be a linked list. You find out when a service that was
fine against shuffled test fixtures meets ordered production data.

If the insertion order can be influenced from outside, this is a denial-of-service vector needing
no cleverness at all: send the keys in order. Fourth structure in this series with that shape,
after NB-02 §3, NB-03 §3 and NB-06 §3.

</details>

***

### Q6. Given sorted data, how do you avoid the disaster?

<details><summary>Answer</summary>

**Four answers, in order of how often they are the right one** (§3.3 measures the last three).

**1. Use a balanced tree.** Red-black, AVL, B-tree — they rebalance on insert and delete and
guarantee $h = O(\log n)$ *whatever* the order. This is `TreeMap`, `std::map`, and NB-08. It is the
answer whenever keys arrive over time, which is most of the time.

**2. Shuffle the input**, if you have all the keys up front and do not need them in order. One
line, and §3.3 measured it taking the height from 19,999 to 34. This is a treap's idea done by
hand; a treap gets the same effect with random priorities and no need for the keys in advance.

**3. Build it balanced directly**, if the keys are already sorted: recursively take the median as
the root. $\Theta(n)$, and it produces a *perfect* tree — §3.3 measured height 14 against the
theoretical optimum of 14. Better than shuffling, and it needs the data up front.

**4. Ask whether you need a tree.** If you never do an ordered operation — no range queries, no
successor, no sorted iteration, no min/max — a hash table (NB-03) gives $O(1)$ expected and cannot
degenerate this way. Q4's table is the decision.

**The non-answer** is "the input will probably be random", which is what an unbalanced BST silently
assumes and what §3 measures the cost of.

</details>

***

### Q7. Find the inorder successor without parent pointers.

<details><summary>Answer</summary>

**The successor is the node at your last left turn.**

Walk down as if searching for the target:

- if the node's key is **greater** than the target, it is a *candidate* — record it and go **left**,
  hoping for something still greater but smaller;
- if the node's key is $\le$ the target, nothing here or to its left can be the successor — go
  **right**.

The answer is the last candidate recorded. $O(h)$, one pass, no parent pointers. The predecessor is
the mirror image: record on going right, take the last right turn.

**Why it works:** every candidate is greater than the target, and each subsequent candidate is
smaller than the previous one (you only record after moving left), so the last is the smallest key
greater than the target — which is the definition.

**Where it shows up:** `TreeMap.higherKey` and `ceilingKey`, in-order iteration without a stack,
and §1.2's delete — though there the successor was known to live in the right subtree, which is the
easy sub-case.

**With parent pointers** the classic formulation is different: if there is a right subtree, take its
leftmost node; otherwise climb until you come up from a *left* child. Worth knowing both, since
whether nodes have parent pointers changes the answer entirely.

</details>

***

### Q8. LCA in a BST versus in a plain binary tree.

<details><summary>Answer</summary>

**A clean measurement of what the invariant is worth.**

**Plain binary tree** (NB-06 Practice 4): a postorder pass returning "did I find one of them?",
$\Theta(n)$ time, and fiddly — you must handle the case where a node *is* one of the targets, and
the case where a target is absent.

**BST** (§2.3): descend from the root, and

- if **both** keys are less than the current node, go left;
- if **both** are greater, go right;
- **otherwise they split here** — this node is the LCA.

$O(h)$, iterative, about three comparisons per level. The reason it collapses to this is that the
paths to $p$ and $q$ diverge exactly once, and the invariant tells you *where* without walking
either path.

**Sharp edges worth naming:**

- **A node is its own ancestor** by the usual definition, so if $p$ is an ancestor of $q$ the answer
  is $p$ — which the "otherwise" branch handles for free, since $p$ does not split.
- **Absent keys.** The descent returns *something* whether or not the keys exist. If the contract
  requires them present, check first; if not, define what you mean. §2.3's reference deliberately
  handles this by building the path you *would* take.
- **Order.** Normalise $p \le q$ at the top or write four comparisons instead of two.

</details>

***

### Q9. Range query and $k$th smallest — how do you avoid $\Theta(n)$?

<details><summary>Answer</summary>

**Range query $[lo, hi]$: prune.** At a node with key $x$, if $x \le lo$ then the entire left
subtree is out of range — do not descend. Symmetrically, if $x \ge hi$ skip the right subtree. Cost
is $O(h + k)$ for $k$ keys reported.

§2.4 counted node visits on a 20,000-key tree to confirm the pruning is real: a range returning 8
keys touched **25 nodes**; one returning 1,004 touched **1,020**. The overhead above $k$ is the
descent, which is $O(h)$.

**$k$th smallest: two answers, and the second is the real one.**

- **Inorder, stopped early:** $O(h + k)$. Fine for small $k$, $\Theta(n)$ when $k \approx n$.
- **Augment the tree:** store each subtree's **size** in its root. Then the $k$th smallest is a
  descent — if `size(left) == k-1` this node is the answer; if `size(left) >= k` go left; else go
  right seeking the $(k - size(left) - 1)$th. **$O(h)$ regardless of $k$.**

The augmentation is the transferable idea, and it is worth naming: **store a summary of each
subtree at its root, maintain it on every update, and queries that would be $\Theta(n)$ become
$O(h)$.** That gives order-statistic trees (size), interval trees (max endpoint), and it is the same
principle as NB-12's Fenwick and segment trees. The cost is that every insert and delete must
update the summary along the path it touched — which is $O(h)$ anyway, so it is free.

</details>

***

### Q10. When should you *not* use a BST?

<details><summary>Answer</summary>

- **When you never need order.** Q4's table: a hash table gives $O(1)$ expected against $O(\log n)$,
  and if you only ever do lookup/insert/delete it wins outright. Using a tree "because it is
  sorted" when nothing reads the order is paying for a feature you do not use.
- **When the tree is unbalanced and you cannot control the input** (§3). Use a balanced tree, or
  a hash table, but not an unbalanced BST fed data you do not choose.
- **When the data does not fit in memory.** A BST's node-per-key pointer chasing is disastrous
  against disk or SSD latency. **B-trees** (NB-08) exist for exactly this: high fan-out means few
  levels means few block reads, which is why every database index is a B-tree and none is a BST.
- **When you need the top $k$ repeatedly rather than order.** A **heap** (NB-09) gives $O(1)$ peek
  at the extreme and $O(\log n)$ pop, using an array with no pointers at all.
- **When keys have no total order.** A BST needs comparison. Hashing needs only equality, so it can
  key on things that cannot be ordered sensibly.
- **When you need worst-case guarantees under concurrency.** Balanced trees are difficult to make
  concurrent; skip lists (NB-04 Challenge 2) are much easier and give the same expected bounds,
  which is why Java has `ConcurrentSkipListMap` and no concurrent red-black tree.

**And the memory point**, which is usually forgotten: a BST node carries two pointers plus an object
header per key. NB-04 §1.1 measured 56 bytes for a two-pointer node in Python against 8 for a list
slot, and NB-04 §3.1 measured what the resulting pointer chasing costs — about 30× against a
contiguous array in Java. A sorted array with binary search beats a BST on both memory and locality
whenever the data is static.

</details>

***

### Q11. Compare BST, hash table and sorted array.

<details><summary>Answer</summary>

| | Sorted array | BST (unbalanced) | Balanced tree | Hash table |
|---|---|---|---|---|
| search | $O(\log n)$ | $O(h)$, worst $\Theta(n)$ | $O(\log n)$ | $O(1)$ expected |
| insert / delete | $\Theta(n)$ (shifting) | $O(h)$ | $O(\log n)$ | $O(1)$ expected |
| sorted iteration | free | $\Theta(n)$ | $\Theta(n)$ | $\Theta(n\log n)$ |
| range query | $O(\log n + k)$ | $O(h + k)$ | $O(\log n + k)$ | $\Theta(n)$ |
| memory per key | **1 slot** | 1 node + 2 pointers | + balance metadata | + empty slots |
| locality | **excellent** | poor | poor | good |
| worst case | guaranteed | **$\Theta(n)$** | guaranteed | $\Theta(n)$ adversarial |

**The decision, in three questions:**

1. **Is the data static?** Then a **sorted array** with binary search wins almost everything —
   same $O(\log n)$ search, a fraction of the memory, and vastly better locality. People reach for
   trees when an array would do because "tree" sounds like the answer to "sorted lookup".
2. **Do you need order at all?** No → **hash table**. Yes → tree.
3. **Do keys arrive over time from a source you do not control?** → **balanced** tree, not a BST.

Note the pattern across the last four notebooks: the worst-case column is the one that decides, and
it is decided by **who supplies the input.** A hash table is $\Theta(n)$ under adversarial keys
(NB-03 §3); an unbalanced BST is $\Theta(n)$ under sorted keys (§3). Only the balanced tree and the
sorted array promise something unconditionally.

</details>

***

### Q12. `TreeMap` vs `HashMap`, and what breaks a `TreeSet`.

<details><summary>Answer</summary>

**`TreeMap` is a red-black tree; `HashMap` is a hash table.** `HashMap` is faster for plain
get/put and should be the default. `TreeMap` exists for the ordered operations `HashMap` cannot
offer at any speed: `firstKey`, `lastKey`, `floorKey`, `ceilingKey`, `higherKey`, `subMap`,
`headMap`, `tailMap`, and sorted iteration.

Because it rebalances, it also **shrugs off the sorted insertion that destroys a hand-rolled BST** —
§1.4 measured the hand-rolled tree at ~9,500 comparisons per search on 20,000 sorted keys while
`TreeMap` stays around 15.

**And now the contract.** `compareTo`/`compare` must be **consistent with `equals`**:
`compare(a,b) == 0` exactly when `a.equals(b)`. Nothing enforces it, and §1.4 shows what breaking it
costs. A `TreeSet` ordered by string length alone, given `"aaa"`, `"bbb"`, `"cccc"`:

```
contents        : [aaa, cccc]
size            : 2          <- "bbb" silently dropped
contains("bbb") : true       <- for a string that is not in the set
```

Nothing is broken in `TreeSet`. An ordered container locates elements by **comparison**, because
that is the only tool it has, so `compare == 0` *is* its definition of "same element". We told it
`"aaa"` and `"bbb"` are the same, and it believed us.

**The rules:** a comparator for a *container* must be a total order consistent with `equals` — break
ties with `thenComparing(naturalOrder())`. A comparator for *sorting* is held to a lower standard,
since arbitrary ordering among equals loses nothing. And note this is **NB-03 §1.6's
`hashCode`/`equals` contract with different methods**: violate the agreement between "how do I
locate it" and "is it the same thing", and the container misfiles or loses entries, silently, with
no exception.

</details>

***

## Coding challenges

### Challenge 1 — an order-statistic tree

Q9 described the augmentation. Build it.

1. Add a `size` field to each node, maintained on every insert and delete.
2. Implement `select(k)` — the $k$th smallest — in $O(h)$, and `rank(key)` — how many keys are
   smaller — also in $O(h)$.
3. Verify both against a sorted list over thousands of randomised operation sequences, with a
   `check_invariant` confirming every node's `size` equals its actual subtree size after **every**
   operation. That invariant is the whole difficulty: the update paths are easy to get subtly wrong
   on the delete cases.
4. Measure `select(n/2)` against the stop-early inorder walk and confirm $O(h)$ against $\Theta(n)$.

### Challenge 2 — a treap, and beat the sorted-input disaster

§3.3 said shuffling is a manual treap. Build the real thing.

1. Each node gets a random **priority**. Maintain the BST invariant on keys *and* the heap invariant
   on priorities, restoring the latter with rotations after insert and delete.
2. Verify against a sorted set, with both invariants asserted after every operation.
3. Feed it **sorted** keys — the input that produces height 19,999 in §3.3 — and measure the height.
   It should look like the random row, because the priorities are random even though the keys are
   not.
4. Explain why that works, and why it needs no knowledge of the keys in advance. This is NB-08's
   subject reached by the easiest possible route: randomisation instead of bookkeeping.

### Challenge 3 — measure the delete-bias folklore

§1.2 mentioned that always deleting via the successor is said to cause leftward imbalance.
Test it.

1. Build a random BST of 10,000 keys, then run a long alternating workload of random inserts and
   deletes, always using the successor.
2. Track the height and the left/right subtree size ratio over time. Does it drift?
3. Repeat, alternating successor and predecessor.
4. Report what you find, including "no measurable difference" if that is the answer. This notebook
   has had several confident claims fail under measurement; treat this one the same way.

***
# Part 5 - Practice

| # | Exercise | The technique | Difficulty |
|---|---|---|---|
| 1 | Search / insert / delete from scratch | The three delete cases | ★★☆☆☆ |
| 2 | Validate a BST | The interval argument | ★★☆☆☆ |
| 3 | $k$th smallest, two ways | Early-stopped inorder, then augmentation | ★★★☆☆ |
| 4 | Recover a BST with two swapped nodes | Inorder as a detector | ★★★★☆ |
| 5 | Trim a BST to a range | Pruning that returns a tree | ★★★☆☆ |
| 6 | Sorted array → balanced BST | §3.3's fix, implemented | ★★☆☆☆ |
| 7 | BST → sorted doubly linked list, in place | Inorder with pointer surgery | ★★★★☆ |
| 8 | Count BSTs of $n$ nodes | Catalan numbers | ★★★★☆ |

***

### 1. Search, insert and delete from scratch

- **Brief:** write all three iteratively (Part 0's rule). Delete is the exercise; the other two are
  warm-ups.
- **Good result:** differential-tested against a sorted set over thousands of randomised operation
  sequences, with the invariant checked after each step — §1.2's harness.
- **The trap:** the **two-children** delete case, and specifically updating the *successor's*
  parent, which may be the deleted node itself when the successor is its immediate right child.
  That is the case hand-written tests miss, and the one a randomised test finds instantly.

### 2. Validate a BST

- **Brief:** write the wrong version (children only) and both right versions (intervals, inorder).
- **Good result:** all three run against each other on randomly shaped trees; the wrong one is
  caught within seconds (§2.1).
- **The trap:** using `float("-inf")` for the initial bounds, which silently restricts you to
  numeric keys; and, in the inorder version, collecting keys then sorting them, which is vacuously
  true and tests nothing.

### 3. $k$th smallest, two ways

- **Brief:** early-stopped inorder first ($O(h+k)$), then the size-augmented version ($O(h)$).
- **Good result:** both verified against `sorted(keys)[k-1]`; the augmented one with an invariant
  asserting every node's stored size matches reality.
- **The trap:** maintaining `size` through **delete**, especially the two-children case where the
  key changes but the structure changes elsewhere. Also: $k$ out of range should be defined, not
  crash.

### 4. Recover a BST with two nodes swapped

Exactly two nodes' values have been exchanged; restore the tree without changing its shape.

- **Brief:** inorder must be sorted, so walk it and find the violations. **Two adjacent nodes
  swapped** produces *one* descent; **two distant nodes** produce *two*. Take the first element of
  the first descent and the second element of the last.
- **Good result:** $\Theta(n)$ time, $O(h)$ space, verified by swapping two random nodes in random
  trees and checking recovery.
- **The trap:** the adjacent case. Code that assumes two violations fails on it, and it is easy to
  miss because most random swaps are distant. Generate adjacent swaps deliberately.

### 5. Trim a BST to a range

Remove every key outside $[lo, hi]$, returning the root of the trimmed tree.

- **Brief:** if a node's key is below `lo`, the whole left subtree goes too — return the trimmed
  right subtree in its place. Symmetrically above `hi`. Otherwise trim both children.
- **Good result:** $O(n)$ worst case, verified by comparing the inorder of the result against the
  filtered inorder of the original.
- **The trap:** *returning* the replacement rather than mutating in place. This is the shape that
  makes the recursion trivial and the iterative version awkward — a good example of when recursion
  genuinely is the clearer tool (NB-06 Q11), provided the height is bounded.

### 6. Sorted array to balanced BST

- **Brief:** §3.3's fix — the middle element is the root, recurse on both halves.
- **Good result:** $\Theta(n)$, height exactly $\lfloor\log_2 n\rfloor$; verify the inorder matches
  the input and the height matches the optimum.
- **The trap:** none in the algorithm, which is the point — this is the *cheap* fix for sorted data
  and people still insert one by one. Then answer the follow-up: what if the input is a sorted
  **linked list** and you may not index it? (Convert to an array, or do an inorder-order
  construction that consumes the list as it goes — the second is much nicer and much harder.)

### 7. BST to a sorted doubly linked list, in place

Reuse `left` as `prev` and `right` as `next`; no new nodes.

- **Brief:** an inorder walk that relinks as it goes, keeping a pointer to the previously visited
  node.
- **Good result:** $\Theta(n)$, no allocation; verify the list order matches the inorder and that
  it is traversable in **both** directions (NB-04 §1.3's invariant).
- **The trap:** you must capture `node.right` **before** overwriting it — the exact aliasing
  discipline of NB-04 §2.1. This problem is the intersection of this notebook and that one, and it
  is a fair interview question precisely because it tests both.

### 8. Count the BSTs of $n$ distinct keys

- **Brief:** choose a root, and the counts of the two subtrees multiply:
  $C_n = \sum_{i=0}^{n-1} C_i C_{n-1-i}$ — the **Catalan numbers**.
- **Good result:** DP in $\Theta(n^2)$; verify against brute-force enumeration for $n \le 8$ by
  actually generating the trees.
- **The trap:** the question is sometimes "how many *shapes*" and sometimes "how many BSTs of these
  keys" — they are the same number, which is worth understanding rather than noticing. Then extend
  to *generating* all of them (NB-17's backtracking), which is where it stops being a formula.

***
# Part 6 - Reading

## Start here

**1. *Introduction to Algorithms* (CLRS), chapter 12 — "Binary Search Trees".**
> The reference treatment. 12.1 proves inorder yields sorted order, 12.3 does delete with the three
> cases drawn out, and **12.4 derives the $O(\log n)$ expected height of a randomly built BST** —
> the result §3.1 measures and finds converges slowly. Read 12.4 alongside the measurement; the
> theorem is about the *expectation*, and the constant takes a long time to show up.

**2. [`TreeMap` Javadoc](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/TreeMap.html)
and [`Comparable` Javadoc](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Comparable.html)** —
**Free.**
> `Comparable`'s documentation contains the sentence §1.4 is built on: *"It is strongly recommended
> (though not required) that natural orderings be consistent with equals"*, followed by an
> explanation of exactly what a sorted set does when you ignore it. Reading the warning **after**
> seeing `contains("bbb") == true` makes it land.

**3. [Sedgewick & Wayne, *Algorithms* 4th ed., §3.2](https://algs4.cs.princeton.edu/32bst/)** —
**Free online.**
> The best visual treatment of BST shape. Their tree-drawing figures for random versus sorted
> insertion are the picture behind §3.1's table, and the section flows directly into red-black
> trees, which is the right next step.

## The source behind each section

| Section | Where it comes from | Free? |
|---|---|---|
| 1.1 — the invariant | **CLRS ch. 12.1** | 🔍 |
| 1.2 — delete's three cases | **CLRS ch. 12.3**; the successor argument is Theorem 12.2 | 🔍 |
| 1.3 — inorder is the invariant | **CLRS ch. 12.1**, Theorem 12.1 | 🔍 |
| 1.4 — the comparison contract | **`Comparable`/`Comparator` Javadoc**; **Bloch, *Effective Java*** item 14 | ✅ (Javadoc) |
| 2.1 — validation | Folklore; the interval formulation is CLRS exercise 12.1-4 | 🔍 |
| 2.4 — augmentation | **CLRS ch. 14**, "Augmenting Data Structures" — order-statistic and interval trees | 🔍 |
| 3.1 — random BST height | **CLRS ch. 12.4** for $O(\log n)$ expected; **Devroye**, *A note on the height of binary search trees*, JACM 1986, for the $4.311\ln n$ constant; **Reed**, *The height of a random binary search tree*, JACM 2003, for the refinement | 🔍 |
| 3.3 — treaps | **Seidel & Aragon**, *Randomized search trees*, Algorithmica 1996 | 🔍 |
| Practice 8 — Catalan numbers | **Stanley**, *Enumerative Combinatorics* vol. 2, ch. 6 | 🔍 |

**Legend:** ✅ free at the link · 🔍 search the exact title on
[Google Scholar](https://scholar.google.com)

### If you read only one

**CLRS chapter 12.4**, on the expected height of a randomly built BST — but read it *with* §3.1's
table open.

The theorem says the expected height is $O(\log n)$, and the sharp constant (Devroye) is
$4.311 \ln n$. §3.1 measures $h/\ln n$ at 3.00, 3.31 and 3.44 for $n$ of a thousand, ten thousand
and a hundred thousand. The asymptotic is correct and the convergence is slow enough that quoting
it as *the height* at realistic sizes overstates it by about a quarter.

That gap is the most useful thing in this reading list, and it is not really about trees:
**an asymptotic result tells you the shape of the curve, not the value at your $n$.** The same
caution applies to every $O(\cdot)$ in this series, and it is why the notebooks measure rather than
quote.

Then read the **`Comparable` Javadoc** — two minutes — and **Sedgewick §3.2** for the pictures.

***
# Appendix

| Symptom | Cause | Fix |
|---|---|---|
| Every operation is $\Theta(n)$ though the code is right | Sorted insertion made a linked list (§3) | Balanced tree (NB-08); or shuffle; or build from the median |
| Fine in tests, slow in production | Fixtures were shuffled, real data was ordered (§3.2) | Test with sorted input on purpose |
| Search misses a key that is in the tree | The invariant is broken — a key sits across an ancestor's boundary (§1.1) | Validate with intervals, not parent comparisons |
| `is_bst` accepts a broken tree | Compared nodes with children only (§2.1) | Interval or inorder validator |
| Validator passes trivially | Collected the inorder then sorted it (§2.1) | Compare each key with the **previous** one |
| Validator breaks on string keys | `float("-inf")` as the initial bound (§2.1) | `None` means unbounded |
| Delete corrupts the tree | Two-children case: successor's parent not updated (§1.2) | Track `succ_parent`; note it may *be* the deleted node |
| Duplicate key present but unfindable | Mixed duplicate conventions (§1.1) | Pick one: reject, count, or one-sided with `<=` |
| `TreeSet` loses elements | Comparator inconsistent with `equals` (§1.4) | `thenComparing(naturalOrder())` to break ties |
| `contains(x)` true for an absent `x` | Same cause — `compare == 0` means "same element" (§1.4) | Same fix |
| `RecursionError` on insert or delete | Recursive implementation on a degenerate tree (Part 0) | Write them iteratively |
| Range query is $\Theta(n)$ | Walking the whole tree and filtering (§2.4) | Prune subtrees that cannot intersect the range |
| $k$th smallest slow for large $k$ | Early-stopped inorder is $O(h + k)$ (§2.4) | Augment nodes with subtree size |

## Checklist for BST code

- [ ] Can the insertion order be sorted or near-sorted (§3)? If so, why is this not a balanced tree?
- [ ] Is the insertion order influenced by anything outside your control (§3.2)?
- [ ] Is the invariant validated with **intervals or inorder**, never parent comparisons (§2.1)?
- [ ] Is the duplicate-key convention written down (§1.1)?
- [ ] Is delete's two-children case tested against a reference, including the successor-is-the-
      immediate-right-child case (§1.2)?
- [ ] Are insert and delete **iterative**, given the tree may degenerate (Part 0)?
- [ ] In Java: is every comparator consistent with `equals` (§1.4)?
- [ ] Do range queries **prune**, or do they scan and filter (§2.4)?
- [ ] Would a **hash table** do — is any ordered operation actually used (§1.3)?
- [ ] Is the data static? If so, would a **sorted array** be better on memory and locality (Q11)?

## Where to go next

| Notebook | Why it follows |
|---|---|
| `balanced_trees_zero_to_hero.ipynb` | The answer to §3 — rebalancing that makes $h = O(\log n)$ unconditionally, and B-trees for disk |
| `heaps_zero_to_hero.ipynb` | A different tree invariant: partial order, in an array, for the extreme rather than the order |
| [`trees_zero_to_hero.ipynb`](trees_zero_to_hero.ipynb) | The traversals this notebook uses, and why everything here is iterative |
| [`hashing_zero_to_hero.ipynb`](hashing_zero_to_hero.ipynb) | The structure to use when you do **not** need order (§1.3's table) |

See [`README.md`](README.md) for the full roster and reading order.